# Módulo 01 — Notebook 3: Git e Versionamento de Projetos ML

---

Você já aprendeu a escrever bom código em Python para ML. Agora vamos aprender a **não perder esse trabalho** — e mais importante, a **reproduzir qualquer resultado** que você já obteve, mesmo meses depois.

### O problema real

Projetos de ML sem versionamento terminam assim:

```
modelo_final.pkl
modelo_final_v2.pkl
modelo_final_v2_corrigido.pkl
modelo_ESSE_MESMO.pkl
modelo_copia_seguranca.pkl
modelo_final_v3_agora_vai_pfvr.pkl
```

E na hora de explicar qual hiperparâmetro gerou aquele resultado de 87% que você apresentou semana passada... não se lembra mais.

### Objetivos deste notebook
1. Dominar o fluxo básico de **Git** para projetos de ciência de dados
2. Adotar uma **estrutura de pastas** padronizada para projetos ML
3. Configurar o **`.gitignore`** correto (o que NÃO versionar)
4. Registrar **experimentos e métricas** de forma simples e versionada
5. Entender quando usar **branches** para separar experimentos
6. Gerenciar ambientes isolados com **conda** e versioná-los junto ao projeto

## 1. Por que ML precisa de versionamento especial?

Em desenvolvimento de software tradicional, você versiona **código**. Em ML, você precisa versionar três coisas que mudam de forma independente:

| O que muda | Exemplo | Como versionar |
|------------|---------|----------------|
| **Código** | trocar SVM por Random Forest | Git (como sempre) |
| **Dados** | adicionar 500 novos exemplos | Git LFS / DVC (ferramentas externas) |
| **Modelo** | weights treinados (.pkl, .h5) | NÃO versionar no Git — armazenar separado |
| **Hiperparâmetros + Métricas** | `C=10, acc=0.87` | Arquivos JSON/YAML versionados no Git |

Neste notebook focamos no que você pode fazer **só com Git**, que já resolve 80% dos problemas.

## 2. Estrutura de Pastas Padronizada para Projetos ML

Antes do Git, precisamos de uma estrutura de projeto que faça sentido. A referência mais usada é o **Cookiecutter Data Science**. Adaptada para o nosso contexto:

In [ ]:
ESTRUTURA_PROJETO = """
meu-projeto-ml/
│
├── data/
│   ├── raw/          ← dados originais (nunca modifique!)
│   ├── processed/    ← dados após pré-processamento
│   └── external/     ← dados de fontes externas
│
├── notebooks/        ← Jupyter notebooks de exploração
│   ├── 01_eda.ipynb
│   └── 02_experimentos.ipynb
│
├── src/              ← código Python reutilizável (não notebooks)
│   ├── __init__.py
│   ├── features.py   ← extração de features
│   ├── models.py     ← definição dos modelos
│   └── evaluate.py   ← métricas e avaliação
│
├── models/           ← modelos treinados salvos (.pkl, .h5)
│                       (NÃO versionar no git — muito grandes)
│
├── experiments/      ← registros de experimentos em JSON/YAML
│   ├── exp_001.json
│   └── exp_002.json
│
├── reports/
│   └── figures/      ← gráficos gerados para relatórios
│
├── .gitignore        ← o que NÃO vai pro git
├── requirements.txt  ← dependências Python
├── README.md         ← documentação do projeto
└── config.yaml       ← hiperparâmetros e configurações
"""
print(ESTRUTURA_PROJETO)

In [ ]:
import os
from pathlib import Path

def criar_estrutura_projeto(nome_projeto: str, caminho_base: str = "."):
    """
    Cria a estrutura de pastas padrão para um projeto de ML.
    Pode ser usada como template para novos projetos.
    """
    raiz = Path(caminho_base) / nome_projeto
    
    pastas = [
        "data/raw", "data/processed", "data/external",
        "notebooks",
        "src",
        "models",
        "experiments",
        "reports/figures",
    ]
    
    for pasta in pastas:
        (raiz / pasta).mkdir(parents=True, exist_ok=True)
        # Criar .gitkeep para que pastas vazias sejam rastreadas pelo git
        (raiz / pasta / ".gitkeep").touch()
    
    # Criar __init__.py para o módulo src
    (raiz / "src" / "__init__.py").touch()
    
    print(f"✓ Estrutura criada em: {raiz.resolve()}")
    for pasta in sorted(pastas):
        print(f"  {raiz / pasta}")
    
    return raiz


# Criar um projeto de exemplo
raiz_projeto = criar_estrutura_projeto("exemplo-projeto-ml")

## 3. Git — Fluxo de Trabalho Básico

### 3.1 Conceitos Fundamentais

```
Área de Trabalho  →  Staging Area  →  Repositório Local  →  Repositório Remoto
 (seus arquivos)       (git add)        (git commit)          (git push)
```

- **Working Directory**: onde você edita os arquivos
- **Staging Area (Index)**: seleção do que vai no próximo commit
- **Repositório Local**: histórico de commits na sua máquina
- **Repositório Remoto**: GitHub / GitLab / Bitbucket

### 3.2 Os Comandos Essenciais

In [ ]:
# Você pode rodar comandos de shell no Jupyter com o prefixo !
# Os exemplos abaixo funcionam dentro do seu terminal ou aqui no notebook

# Navegar para o projeto criado
import os
os.chdir("exemplo-projeto-ml")

# 1. Inicializar o repositório
!git init
print("")

# 2. Ver o estado atual (sem nenhum arquivo rastreado ainda)
!git status

In [ ]:
import json
from pathlib import Path

# Criar arquivos iniciais do projeto

# README.md
Path("README.md").write_text("""# Exemplo Projeto ML

Projeto de demonstração de boas práticas de versionamento.

## Como executar
```bash
pip install -r requirements.txt
jupyter notebook notebooks/
```
""")

# requirements.txt
Path("requirements.txt").write_text("""numpy>=1.24.0
pandas>=2.0.0
scikit-learn>=1.3.0
matplotlib>=3.7.0
pyyaml>=6.0
""")

# config.yaml — hiperparâmetros centralizados
config = {
    "dados": {
        "taxa_amostragem": 22050,
        "duracao_maxima": 5.0,
        "test_size": 0.15,
        "val_size": 0.15,
        "random_state": 42,
    },
    "features": {
        "n_mfcc": 13,
        "n_janela": 512,
        "n_salto": 256,
    },
    "modelo": {
        "tipo": "SVM",
        "C": 10,
        "kernel": "rbf",
        "gamma": "scale",
    }
}

# Usar try/except pois pyyaml pode não estar instalado
try:
    import yaml
    Path("config.yaml").write_text(yaml.dump(config, default_flow_style=False, allow_unicode=True))
    print("config.yaml criado com pyyaml")
except ImportError:
    # Fallback: salvar como JSON
    Path("config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False))
    print("config.json criado (pyyaml não disponível)")

print("✓ Arquivos iniciais criados")

In [ ]:
# Criar o .gitignore — fundamental!
GITIGNORE_ML = """# ==========================================
# .gitignore para projetos de Machine Learning
# ==========================================

# --- Python ---
__pycache__/
*.py[cod]
*.pyo
.Python
env/
venv/
.venv/
*.egg-info/

# --- Jupyter ---
.ipynb_checkpoints/
*.ipynb_checkpoints

# --- Conda (ambientes locais — NÃO versionar a pasta do ambiente) ---
# O environment.yml SIM deve ir pro git (é o "receita" do ambiente)
.conda/
conda-env/
envs/

# --- Dados (geralmente grandes demais para o git) ---
data/raw/*
data/processed/*
data/external/*
!data/**/.gitkeep     # manter estrutura de pastas
*.wav
*.mp3
*.flac
*.csv
*.parquet
*.h5
*.hdf5

# --- Modelos treinados (binários grandes) ---
models/*
!models/.gitkeep
*.pkl
*.joblib
*.pt
*.pth
*.onnx
*.pb

# --- Credenciais e configurações locais ---
.env
.env.local
*.key
secrets.yaml

# --- IDEs e editores ---
.vscode/
.idea/
*.swp
*.swo
.DS_Store
Thumbs.db

# --- Logs e temporários ---
logs/
*.log
tmp/
temp/
"""

Path(".gitignore").write_text(GITIGNORE_ML)
print("✓ .gitignore criado")
print("\nConteúdo do .gitignore:")
print(GITIGNORE_ML)

In [ ]:
# Configurar identidade (apenas uma vez por máquina)
# Substitua pelos seus dados
!git config user.name "Seu Nome"
!git config user.email "seu@email.com"

# Adicionar arquivos à staging area
!git add README.md requirements.txt .gitignore

# Tentar adicionar config.yaml ou config.json dependendo do que foi criado
!git add config.yaml 2>/dev/null || git add config.json 2>/dev/null

# Adicionar estrutura de pastas
!git add data/ src/ notebooks/ experiments/ reports/

print("Estado após git add:")
!git status

In [ ]:
# Primeiro commit!
!git commit -m "feat: estrutura inicial do projeto ML"

print("\nHistórico de commits:")
!git log --oneline

## 4. Commits Semânticos — Mensagens que Fazem Sentido

Uma mensagem de commit ruim: `"arrumei as coisas"`

O padrão **Conventional Commits** resolve isso:

```
<tipo>(<escopo opcional>): <descrição curta>
```

| Tipo | Quando usar | Exemplo |
|------|------------|--------|
| `feat` | nova feature ou notebook | `feat: adicionar extrator de MFCCs` |
| `fix` | correção de bug | `fix: corrigir divisão por zero no ZCR` |
| `data` | mudança nos dados | `data: adicionar 200 amostras de raiva` |
| `exp` | novo experimento | `exp: testar Random Forest com n=500` |
| `refactor` | refatoração sem mudar comportamento | `refactor: extrair ExtractorBase para ABC` |
| `docs` | documentação | `docs: adicionar docstring ao Pipeline` |
| `chore` | manutenção | `chore: atualizar requirements.txt` |

### Por que isso importa em ML?
Quando você olhar o histórico meses depois, vai ver exatamente **o que mudou** entre os experimentos e por quê a acurácia subiu ou desceu.

## 5. Rastreamento de Experimentos com JSON

Sempre que treinar um modelo, **registre o experimento** em um arquivo JSON. É simples, legível e versionável no git.

In [ ]:
import json
import datetime
import numpy as np
from pathlib import Path

def registrar_experimento(
    nome_exp: str,
    hiperparametros: dict,
    metricas: dict,
    notas: str = "",
    pasta: str = "experiments"
) -> Path:
    """
    Salva um registro de experimento em JSON.
    
    Use sempre que treinar um modelo — isso permite reproduzir
    qualquer resultado e comparar experimentos sistematicamente.
    """
    registro = {
        "nome": nome_exp,
        "timestamp": datetime.datetime.now().isoformat(),
        "hiperparametros": hiperparametros,
        "metricas": metricas,
        "notas": notas,
    }
    
    # Nome do arquivo baseado no timestamp (garante unicidade)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    nome_arquivo = f"{pasta}/{ts}_{nome_exp.replace(' ', '_')}.json"
    
    Path(nome_arquivo).write_text(
        json.dumps(registro, indent=2, ensure_ascii=False)
    )
    print(f"✓ Experimento registrado: {nome_arquivo}")
    return Path(nome_arquivo)


# Simular treinamento de modelos e registrar experimentos
np.random.seed(42)

# Experimento 1: SVM com parâmetros padrão
arq1 = registrar_experimento(
    nome_exp="svm_baseline",
    hiperparametros={
        "modelo": "SVM",
        "kernel": "rbf",
        "C": 1.0,
        "gamma": "scale",
        "n_features": 81,
        "n_mfcc": 13,
    },
    metricas={
        "acc_treino": 0.92,
        "acc_val": 0.74,
        "acc_teste": None,   # ainda não avaliado no teste!
        "f1_macro": 0.73,
        "auc_medio": 0.91,
    },
    notas="Baseline com hiperparâmetros default. Possível overfitting (gap treino-val grande)."
)

# Experimento 2: SVM com C maior
import time; time.sleep(1)  # garantir timestamps diferentes
arq2 = registrar_experimento(
    nome_exp="svm_C10",
    hiperparametros={
        "modelo": "SVM",
        "kernel": "rbf",
        "C": 10.0,
        "gamma": "scale",
        "n_features": 81,
        "n_mfcc": 13,
    },
    metricas={
        "acc_treino": 0.95,
        "acc_val": 0.81,
        "acc_teste": None,
        "f1_macro": 0.80,
        "auc_medio": 0.94,
    },
    notas="Aumentar C melhorou validação. Ajuste fino de gamma pode ajudar mais."
)

In [ ]:
import pandas as pd

def comparar_experimentos(pasta: str = "experiments") -> pd.DataFrame:
    """
    Carrega todos os experimentos registrados e os compara em uma tabela.
    """
    registros = []
    for arquivo in sorted(Path(pasta).glob("*.json")):
        with open(arquivo) as f:
            exp = json.load(f)
        
        linha = {
            "nome": exp["nome"],
            "timestamp": exp["timestamp"][:16],  # só data e hora
        }
        # Achatar hiperparâmetros e métricas em colunas
        linha.update({f"hp_{k}": v for k, v in exp["hiperparametros"].items()})
        linha.update({f"met_{k}": v for k, v in exp["metricas"].items()})
        linha["notas"] = exp["notas"]
        registros.append(linha)
    
    return pd.DataFrame(registros)


df_exps = comparar_experimentos()
colunas_interesse = ["nome", "timestamp", "hp_C", "met_acc_val", "met_f1_macro", "met_auc_medio"]
print("Comparação de experimentos:")
print(df_exps[colunas_interesse].to_string(index=False))

In [ ]:
# Versionar os experimentos no git
!git add experiments/
!git commit -m "exp: svm baseline e svm_C10 — val 74% → 81%"

print("\nHistórico atualizado:")
!git log --oneline

## 6. Branches — Separando Linhas de Experimento

Quando você quer explorar uma ideia nova (ex: trocar SVM por rede neural) sem "estragar" o que já funciona, use um **branch**.

```
main ──────●──────────────────────●──── (código estável)
            \                    /
             ●──●──● experiment/redes-neurais
```

Convenção de nomes para branches em ML:
- `experiment/<descricao>` — explorar nova ideia
- `feature/<nome>` — adicionar funcionalidade ao código
- `fix/<descricao>` — correção de bug
- `data/<descricao>` — mudança no pipeline de dados

In [ ]:
# Ver branches existentes
!git branch

print("")

# Criar e trocar para um novo branch de experimento
!git checkout -b experiment/random-forest

print("")
!git branch

In [ ]:
import time

# Neste branch, experimentamos Random Forest
time.sleep(1)
arq3 = registrar_experimento(
    nome_exp="random_forest_200",
    hiperparametros={
        "modelo": "RandomForest",
        "n_estimators": 200,
        "max_depth": 15,
        "min_samples_leaf": 2,
        "n_features": 81,
        "n_mfcc": 13,
    },
    metricas={
        "acc_treino": 0.99,
        "acc_val": 0.78,
        "acc_teste": None,
        "f1_macro": 0.77,
        "auc_medio": 0.93,
    },
    notas="RF tem acc_treino muito alta → overfitting. Tentar max_depth menor."
)

!git add experiments/
!git commit -m "exp: random forest n=200 — val 78%, sinais de overfitting"

print("\nLog do branch experiment/random-forest:")
!git log --oneline

In [ ]:
# Voltar para o branch principal
!git checkout main 2>/dev/null || git checkout master

print("\nBranch atual:")
!git branch

print("\nExperimentos visíveis no main (sem o RF):")
import glob
for arq in sorted(glob.glob("experiments/*.json")):
    print(f"  {arq}")

In [ ]:
# Quando o experimento for promissor, fazer merge
!git merge experiment/random-forest -m "merge: trazer experimentos de random forest"

print("\nApós o merge, todos os experimentos estão no main:")
for arq in sorted(glob.glob("experiments/*.json")):
    print(f"  {arq}")

print("\nLog completo:")
!git log --oneline --graph --all

## 7. Reprodutibilidade — A Regra Mais Importante

Um experimento de ML é reproduzível quando qualquer pessoa (ou você mesmo, meses depois) consegue chegar exatamente no mesmo resultado. Isso exige:

1. **Seeds fixas** para todo gerador de números aleatórios
2. **Versões fixas** de todas as bibliotecas
3. **Configurações versionadas** (não hardcoded no código)

##### TEMPLATE: Cabeçalho padrão para notebooks/scripts de ML

In [ ]:
import numpy as np
import random
import os
import sys
import json
import datetime
from pathlib import Path

# --- Seed global ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
# Se usar TensorFlow: tf.random.set_seed(SEED)
# Se usar PyTorch:    torch.manual_seed(SEED)

# --- Carregar configurações do arquivo (não hardcode!) ---
try:
    import yaml
    with open("config.yaml") as f:
        CONFIG = yaml.safe_load(f)
except (ImportError, FileNotFoundError):
    with open("config.json") as f:
        CONFIG = json.load(f)

# --- Registrar versões das bibliotecas ---
import sklearn
import pandas as pd
import matplotlib

versoes = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
}

print(f"{'='*45}")
print(f"Ambiente de execução — {datetime.date.today()}")
print(f"{'='*45}")
for lib, ver in versoes.items():
    print(f"  {lib:15s}: {ver}")
print(f"  seed global   : {SEED}")
print(f"\nConfiguração carregada: {list(CONFIG.keys())}")

In [ ]:
# Gerar requirements.txt com versões EXATAS (para reprodução)
print("Para gerar requirements com versões exatas, rode no terminal:")
print("  pip freeze > requirements.txt")
print("")
print("Para instalar em outro ambiente:")
print("  pip install -r requirements.txt")
print("")

# Simular o que pip freeze mostraria
requirements_exatos = "\n".join([
    f"{lib}=={ver}" for lib, ver in versoes.items()
    if lib != "python"
])
print("Exemplo de requirements.txt com versões fixas:")
print(requirements_exatos)

## 8. Ambientes Isolados com Conda

### Por que ambientes virtuais?

Imagine que seu projeto A precisa de `scikit-learn==1.0` e seu projeto B precisa de `scikit-learn==1.3`. Sem ambiente virtual, instalar um quebra o outro. Ambientes virtuais resolvem isso criando uma **instalação Python isolada por projeto**.

### Conda vs venv — qual usar?

| | `venv` (Python puro) | `conda` |
|--|----------------------|---------|
| Isola pacotes Python | ✓ | ✓ |
| Isola a versão do Python | ✗ | ✓ |
| Gerencia pacotes não-Python (C, CUDA, etc.) | ✗ | ✓ |
| Pré-compila binários otimizados | ✗ | ✓ |
| Arquivo de ambiente | `requirements.txt` | `environment.yml` |
| Quando usar | projetos simples, CI/CD | ciência de dados, ML, áudio |

**Recomendação para bioacústica:** use conda — `librosa`, `scipy` e outras bibliotecas de áudio dependem de compilações C que o conda gerencia automaticamente.

### O que versionar no git

```
✓ environment.yml   → "receita" para recriar o ambiente (commitar sempre)
✗ envs/             → pasta com os binários do ambiente (ignorar no .gitignore)
```

##### Conda na prática — comandos de referência

In [ ]:
COMANDOS_CONDA = """
# --- Criar ambiente para o projeto ---
conda create -n meu-projeto-ml python=3.9
conda activate meu-projeto-ml

# --- Instalar pacotes (prefira conda quando disponível) ---
conda install numpy pandas scikit-learn matplotlib seaborn
conda install -c conda-forge librosa pyyaml

# Pacotes só no PyPI: use pip dentro do ambiente conda
pip install noisereduce praat-parselmouth

# --- Exportar o ambiente para um arquivo (versionar no git!) ---
conda env export > environment.yml

# --- Recriar o ambiente exato em outra máquina ---
conda env create -f environment.yml
conda activate meu-projeto-ml

# --- Outros comandos úteis ---
conda env list                       # listar todos os ambientes
conda list                           # pacotes do ambiente ativo
conda env remove -n meu-projeto-ml  # remover ambiente
conda deactivate                     # sair do ambiente
"""
print(COMANDOS_CONDA)

# Gerar um environment.yml de exemplo (estrutura real)
ENVIRONMENT_YML = """name: meu-projeto-ml
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.9.18
  - numpy=1.24.3
  - pandas=2.0.3
  - scikit-learn=1.3.0
  - matplotlib=3.7.2
  - seaborn=0.12.2
  - scipy=1.11.1
  - jupyter=1.0.0
  - pyyaml=6.0
  - pip:
    - librosa==0.10.1
    - noisereduce==3.0.0
    - praat-parselmouth==0.4.3
"""

Path("environment.yml").write_text(ENVIRONMENT_YML)
print("✓ environment.yml gerado")
print("\nConteúdo do environment.yml:")
print(ENVIRONMENT_YML)

# Commitar no git — environment.yml DEVE ser versionado
!git add environment.yml
!git commit -m "chore: adicionar environment.yml do conda"
print("\n✓ environment.yml versionado no git")

## 9. Fluxo de Trabalho Completo — Resumo Visual

```
NOVO PROJETO:
  git init
  criar_estrutura_projeto("nome")
  git add .gitignore README.md requirements.txt config.yaml environment.yml
  git commit -m "feat: estrutura inicial"
  git remote add origin <url-github>
  git push -u origin main

CADA EXPERIMENTO:
  git checkout -b experiment/descricao
  → editar código ou config.yaml
  → treinar modelo
  → registrar_experimento(nome, hiperparametros, metricas)
  git add src/ experiments/ notebooks/
  git commit -m "exp: descricao — val X%"
  → se bom: git checkout main && git merge experiment/descricao
  → se ruim: git branch -d experiment/descricao  (descartar)

MODELO FINAL:
  → avaliar no conjunto de TESTE
  → atualizar o registro do experimento com acc_teste
  git commit -m "exp: resultado final — teste X%, AUC Y"
  git tag v1.0.0-modelo-emocao
  git push
```

In [ ]:
# Git tags: marcar versões estáveis de modelos
!git tag -a v0.1.0 -m "Baseline: SVM, val=81%, antes da avaliação no teste"

print("Tags do repositório:")
!git tag -l

print("\nLog com tags:")
!git log --oneline --decorate

## 10. Comandos Git de Referência Rápida

| Comando | O que faz |
|---------|----------|
| `git init` | Inicializa repositório na pasta atual |
| `git status` | Mostra arquivos modificados/staged/untracked |
| `git add <arquivo>` | Adiciona arquivo à staging area |
| `git add -p` | Adiciona interativamente (trecho por trecho) |
| `git commit -m "msg"` | Cria commit com mensagem |
| `git log --oneline` | Histórico resumido |
| `git diff` | Diferença entre working dir e último commit |
| `git diff HEAD~1` | Comparar com commit anterior |
| `git checkout -b branch` | Criar e entrar em novo branch |
| `git checkout main` | Voltar para o branch principal |
| `git merge branch` | Trazer alterações de outro branch |
| `git stash` | Guardar mudanças temporariamente sem commit |
| `git stash pop` | Recuperar mudanças guardadas |
| `git tag -a v1.0` | Marcar versão estável |
| `git remote add origin <url>` | Conectar ao repositório remoto |
| `git push -u origin main` | Enviar para o remoto (primeira vez) |
| `git pull` | Baixar e integrar mudanças do remoto |
| `git clone <url>` | Clonar repositório existente |

## 11. Gitmoji.dev - algo mais visual

| Emoji | Comando | O que faz |
|-------|---------|-----------|
|🎉| `:tada:` | Iniciando novo projeto |
|✨| `:sparkles:` | Nova feature |
|♻️| `:recycle:` | Refatoração |
|🐛| `:bug:` | Correção de bug |
|⚡️| `:zap:` | Melhoria de performance |

## Resumo

Com o que vimos neste notebook, você consegue:

- Criar projetos ML organizados que qualquer pessoa entende ao clonar
- Nunca mais perder um resultado — todo experimento está no histórico do git
- Comparar experimentos lado a lado com a função `comparar_experimentos()`
- Explorar ideias novas em branches sem risco de quebrar o que funciona
- Reproduzir qualquer resultado com seeds + config + versões fixas

Esses hábitos fazem a diferença entre um projeto que "funcionou uma vez" e um projeto que **você consegue explicar, apresentar e iterar** com confiança.